In [ ]:
# ==========================================
# TESIS ZMVM: PM, clima y salud
# Autor: Arely Leal
# Descripción: calcular promedios ponderados de 12 horas (NowCast) de material particulado con base en los criterios estipulados en la NOM-172-SEMARNAT-2019. 
# Periodo: 2000-2019
# ==========================================

In [5]:
#Usar para PM2.5

import pandas as pd
import numpy as np
import math
import os

# =========================
# Configuración
# =========================

INPUT_CSV = "AGREGAR RUTA DEL ARCHIVO"
OUTPUT_CSV = os.path.join(
    os.path.dirname(INPUT_CSV),
    "AGREGAR NOMBRE DE SALIDA.csv"
)

TARGET_HOURS = {8, 18}   # 08:00 y 18:00
WINDOW = 12              # ventana móvil de 12 horas

# =========================
# Redondeo (criterio solicitado)
# - primer decimal 0–4: se mantiene
# - primer decimal >=5: entero superior
# =========================
def redondeo_normativo(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    return int(math.floor(float(x) + 0.5))

# =========================
# Función NowCast 12 h
# =========================
def nowcast_12h(values_recent_first: np.ndarray) -> float:
    """
    values_recent_first: arreglo de longitud 12 con el orden:
    [t, t-1, ..., t-11] (más reciente primero). Puede contener np.nan.

    Criterio de faltantes:
    - Debe haber datos válidos en al menos 2 de las 3 horas más recientes (t, t-1, t-2).
    """
    last3 = values_recent_first[:3]
    if np.sum(~np.isnan(last3)) < 2:
        return np.nan

    valid = values_recent_first[~np.isnan(values_recent_first)]
    if valid.size == 0:
        return np.nan

    cmax = np.max(valid)
    cmin = np.min(valid)

    if cmax == 0:
        return 0  # todo cero -> nowcast cero (entero)

    # Factor de ponderación w, acotado a mínimo 0.5
    w_raw = 1.0 - ((cmax - cmin) / cmax)
    w = max(0.5, min(1.0, w_raw))

    # Pesos: más reciente tiene exponente 0, luego 1, ..., 11
    exponents = np.arange(0, WINDOW, dtype=float)
    weights = np.power(w, exponents)

    # Excluir faltantes y renormalizar
    mask = ~np.isnan(values_recent_first)
    if np.sum(mask) == 0:
        return np.nan

    weighted_sum = np.sum(values_recent_first[mask] * weights[mask])
    weight_sum = np.sum(weights[mask])
    if weight_sum == 0:
        return np.nan

    nowcast_value = weighted_sum / weight_sum

    # Redondeo final a entero con tu criterio
    return redondeo_normativo(nowcast_value)

# =========================
# Lectura y preparación
# =========================
df = pd.read_csv(INPUT_CSV)

# Crear datetime (HORA 1–24 -> 0–23)
df["datetime"] = pd.to_datetime(df["FECHA"], errors="coerce") + pd.to_timedelta(df["HORA"] - 1, unit="h")
df = df.dropna(subset=["datetime"]).sort_values("datetime")

# Columnas de municipios (todo lo demás excepto FECHA, HORA, datetime)
municipios = [c for c in df.columns if c not in {"FECHA", "HORA", "datetime"}]

# Asegurar numérico
for m in municipios:
    df[m] = pd.to_numeric(df[m], errors="coerce")

# =========================
# Cálculo NowCast (solo 08:00 y 18:00)
# =========================
results = []

for municipio in municipios:
    serie = df.set_index("datetime")[municipio].sort_index()

    for t in serie.index:
        if t.hour not in TARGET_HOURS:
            continue

        # Ventana de 12h terminando en t
        window_idx = pd.date_range(end=t, periods=WINDOW, freq="h")
        window_vals = serie.reindex(window_idx).to_numpy()

        # Si no hay suficientes timestamps (p.ej. inicio de serie), saltar
        if window_vals.size != WINDOW:
            continue

        # Reordenar: más reciente primero
        values_recent_first = window_vals[::-1]

        nc = nowcast_12h(values_recent_first)

        results.append({
            "municipio": municipio,
            "datetime": t,
            "hora": t.hour,
            "nowcast_PM25_12h": nc
        })

out = pd.DataFrame(results).sort_values(["municipio", "datetime"])

out["nowcast_PM25_12h"] = out["nowcast_PM25_12h"].astype(object)
out.loc[out["nowcast_PM25_12h"].isna(), "nowcast_PM25_12h"] = "NaN"

out.to_csv(OUTPUT_CSV, index=False)
print("Archivo generado:", OUTPUT_CSV)

✅ Listo. Archivo generado: /Users/arelyleal/Downloads/TESIS/BASES DE DATOS/MATERIAL PARTICULADO/PM2.5/nowcast_PM25_12h_8am_6pm_municipios_redondeado.csv


In [9]:
#Usar para PM10

import pandas as pd
import numpy as np
import math
import os

# =========================
# Configuración
# =========================
INPUT_CSV = "AGREGAR RUTA DEL ARCHIVO"
OUTPUT_CSV = os.path.join(
    os.path.dirname(INPUT_CSV),
    "AGREGAR NOMBRE DE SALIDA.csv"
)

TARGET_HOURS = {8, 18}   # 08:00 y 18:00
WINDOW = 12              # ventana móvil de 12 horas

# =========================
# Redondeo (criterio solicitado)
# - primer decimal 0–4: se mantiene
# - primer decimal >=5: entero superior
# =========================
def redondeo_normativo(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    return int(math.floor(float(x) + 0.5))

# =========================
# Función NowCast 12 h
# =========================
def nowcast_12h(values_recent_first: np.ndarray) -> float:
    """
    values_recent_first: arreglo de longitud 12 con el orden:
    [t, t-1, ..., t-11] (más reciente primero). Puede contener np.nan.

    Criterio de faltantes:
    - Debe haber datos válidos en al menos 2 de las 3 horas más recientes (t, t-1, t-2).
    """
    last3 = values_recent_first[:3]
    if np.sum(~np.isnan(last3)) < 2:
        return np.nan

    valid = values_recent_first[~np.isnan(values_recent_first)]
    if valid.size == 0:
        return np.nan

    cmax = np.max(valid)
    cmin = np.min(valid)

    if cmax == 0:
        return 0  # todo cero -> nowcast cero (entero)

    # Factor de ponderación w, acotado a mínimo 0.5
    w_raw = 1.0 - ((cmax - cmin) / cmax)
    w = max(0.5, min(1.0, w_raw))

    # Pesos: más reciente tiene exponente 0, luego 1, ..., 11
    exponents = np.arange(0, WINDOW, dtype=float)
    weights = np.power(w, exponents)

    # Excluir faltantes y renormalizar
    mask = ~np.isnan(values_recent_first)
    if np.sum(mask) == 0:
        return np.nan

    weighted_sum = np.sum(values_recent_first[mask] * weights[mask])
    weight_sum = np.sum(weights[mask])
    if weight_sum == 0:
        return np.nan

    nowcast_value = weighted_sum / weight_sum

    # Redondeo final a entero con tu criterio
    return redondeo_normativo(nowcast_value)

# =========================
# Lectura y preparación
# =========================
df = pd.read_csv(INPUT_CSV)

# Crear datetime (HORA 1–24 -> 0–23)
df["datetime"] = pd.to_datetime(
    df["FECHA"],
    format="%d/%m/%y",
    errors="coerce"
) + pd.to_timedelta(df["HORA"] - 1, unit="h")
df = df.dropna(subset=["datetime"]).sort_values("datetime")

# Columnas de municipios (todo lo demás excepto FECHA, HORA, datetime)
municipios = [c for c in df.columns if c not in {"FECHA", "HORA", "datetime"}]

# Asegurar numérico
for m in municipios:
    df[m] = pd.to_numeric(df[m], errors="coerce")

# =========================
# Cálculo NowCast (solo 08:00 y 18:00)
# =========================
results = []

for municipio in municipios:
    serie = df.set_index("datetime")[municipio].sort_index()

    for t in serie.index:
        if t.hour not in TARGET_HOURS:
            continue

        # Ventana de 12h terminando en t
        window_idx = pd.date_range(end=t, periods=WINDOW, freq="h")
        window_vals = serie.reindex(window_idx).to_numpy()

        # Si no hay suficientes timestamps (inicio de serie), saltar
        if window_vals.size != WINDOW:
            continue

        # Reordenar: más reciente primero
        values_recent_first = window_vals[::-1]

        nc = nowcast_12h(values_recent_first)

        results.append({
            "municipio": municipio,
            "datetime": t,
            "hora": t.hour,
            "nowcast_PM10_12h": nc
        })

out = pd.DataFrame(results).sort_values(["municipio", "datetime"])

out["nowcast_PM10_12h"] = out["nowcast_PM10_12h"].astype(object)
out.loc[out["nowcast_PM10_12h"].isna(), "nowcast_PM10_12h"] = "NaN"

out.to_csv(OUTPUT_CSV, index=False)
print("Archivo generado:", OUTPUT_CSV)

Archivo generado: /Users/arelyleal/Downloads/TESIS/BASES DE DATOS/MATERIAL PARTICULADO/PM10/nowcast_PM10_12h_8am_6pm_municipios_redondeado.csv
